# Bielik on TPU — Episode 2: Matmul on the MXU

Companion notebook for [`bielik-on-tpu`](https://github.com/qooba/bielik-on-tpu) episode 2. Clones the repo, installs it, prints the matmul kernel's real source, and runs its benchmark (including block-size autotuning) on a real TPU.

**Before running**: `Runtime` → `Change runtime type` → select a `TPU` hardware accelerator. The kernel's `interpret=True` mode runs on CPU with no TPU needed, but this notebook's benchmark cell calls the real Pallas/Mosaic path, which does need one.

## 1. Clone the repository

`qooba/bielik-on-tpu` is public — no credentials needed.

In [ ]:
import os

REPO_URL = "https://github.com/qooba/bielik-on-tpu.git"
REPO_DIR = "/content/bielik-on-tpu"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Install the repo as an editable package

`pyproject.toml` declares `kernels`/`benchmarks` as an installable package with no dependencies of its own — `jax[tpu]` needs its own pinned install (next section).

In [ ]:
!pip install -e . --no-deps -q

## 3. Install a matched jax/libtpu pair

Colab's preinstalled `jax`/`libtpu` versions can be mismatched, which makes `pallas_call` fail with a Mosaic version error even though plain JAX ops work. Installing this repo's pinned `jax[tpu]==0.11.1` and restarting the runtime avoids that.

Running the next cell **restarts the Colab runtime** — the "session crashed" message that follows is expected. Continue from the cell after next; no need to re-run the clone/install cells above.

In [ ]:
!pip install -q -U 'jax[tpu]==0.11.1' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

import os

os.kill(os.getpid(), 9)

In [ ]:
%cd /content/bielik-on-tpu

import jax

print(jax.devices())

## 4. Import the kernel and read its source

`kernels/matmul/matmul.py` holds both implementations side by side: `matmul_ref` (plain JAX, `precision=HIGHEST` for fp32 via `dot_precision`) and `matmul_pallas` (the tiled Pallas/TPU kernel: a 3D grid over `M`/`N`/`K` tiles with an fp32 VMEM accumulator). `??` prints the real source straight from the cloned repo.

In [ ]:
from kernels.matmul.matmul import matmul_pallas, matmul_ref

??matmul_ref

In [ ]:
??matmul_pallas

## 5. Quick correctness check

`matmul_pallas` vs `matmul_ref` vs a plain NumPy reimplementation, at a Bielik-shaped `(M, K, N)`, in **bf16** — Bielik-1.5B's real checkpoint dtype, not fp32. Tolerance is relative, not absolute: bf16 itself has ~3 decimal digits of precision, so its own rounding noise dominates over any accumulation-order difference between the two implementations.

In [ ]:
import jax.numpy as jnp
import numpy as np

key = jax.random.PRNGKey(0)
key_a, key_b, key_bias = jax.random.split(key, 3)
M, K, N = 128, 1536, 1536
dtype = jnp.bfloat16

a = jax.random.normal(key_a, (M, K), dtype=dtype)
b = jax.random.normal(key_b, (K, N), dtype=dtype)
bias = jax.random.normal(key_bias, (N,), dtype=dtype)

out_pallas = matmul_pallas(a, b, bias)
out_ref = matmul_ref(a, b, bias)

a_np, b_np, bias_np = (np.asarray(x.astype(jnp.float32)) for x in (a, b, bias))
out_numpy = (a_np @ b_np + bias_np).astype(np.float32)

scale = float(np.abs(out_numpy).max()) + 1e-6
diff_pallas_ref = float(
    jnp.max(jnp.abs(out_pallas.astype(jnp.float32) - out_ref.astype(jnp.float32)))
)
diff_pallas_numpy = float(np.max(np.abs(np.asarray(out_pallas.astype(jnp.float32)) - out_numpy)))

print("max|pallas - ref|   / scale =", diff_pallas_ref / scale)
print("max|pallas - numpy| / scale =", diff_pallas_numpy / scale)

## 6. Run the benchmark

`pallas` (fixed `block=128`) vs `pallas_autotuned` (real-hardware block-size search) vs `jax_native`, real TFLOPS numbers on this TPU session. `benchmark_matmul.py` has no single `run_benchmark()` wrapper the way episode 1's RMSNorm script does — its `__main__` block calls two real functions directly, `_run_size_sweep()` (square `M=K=N` sweep) and `run_bielik_shapes()` (the eight named Bielik projection shapes: Q/K/FFN-up/FFN-down × single-token/`seq_len=128`) — so this notebook calls the same two functions the same way.

In [ ]:
from benchmarks.matmul.benchmark_matmul import _run_size_sweep, run_bielik_shapes

_run_size_sweep([512, 1024, 2048, 4096], "docs/plots/matmul/")
run_bielik_shapes(save_path="docs/plots/matmul/")

## 7. View the plots

In [ ]:
from IPython.display import Image, display

for name in [
    "matmul-tflops-vs-size.png",
    "matmul-summary-bielik-shapes-latency_(ms).png",
    "matmul-summary-bielik-shapes-tflops.png",
]:
    display(Image(filename=f"docs/plots/matmul/{name}"))